# Optimization Campaign (HITL)

**Prerequisites:** TermNorm backend at `http://127.0.0.1:8000` | Groq API key in `.env` | Restart kernel after first sync

**Workflow:** Setup → Data → Explore → Optimize → Results

In [ ]:
%load_ext autoreload
%autoreload 2

from campaign_lib import *

# --- Services ---
svc = await init_services()
TASK_DESCRIPTION = load_task_description(
    r"C:\Users\dsacc\OfficeAddinApps\TermNorm-excel\backend-api\config\LCA_INPUT_PATTERNS.md"
)

# --- Campaign config ---
# ALL experiment knobs live here — no hidden defaults in service code.
campaign_config = {
    "sample_size": 15,              # queries per eval step (0 = all)
    "exploration_sample_size": 10,  # queries per scan/grid point (can be smaller)
    "exploration_rate": 0.5,        # PRIMARY KNOB: 0.0=conservative, 1.0=aggressive
    "exclude_nodes": ["llm_ranking"],    # nodes to skip (e.g. ["entity_profiling"])
    # --- Backend node overrides ---
    # Override values from GET /pipeline. Flat names resolved via schema override_map.
    # See show_pipeline_snapshot() output for available params per node.
    "pipeline_overrides": {
        "ranking_model": "openai/gpt-oss-120b",     # llm_ranking LLM model
        "profiling_model": "openai/gpt-oss-120b",   # entity_profiling LLM model
        "profiling_max_tokens": 4000,             # entity_profiling max output tokens
    },
    "optimization": {
        # --- Core loop ---
        "patience": 2,                   # consecutive non-improvements before stop/escalate
        "max_rounds": None,              # None = unlimited
        "n_variants": 5,                 # candidates per round
        "creativity": 0.7,               # temperature for candidate generation
        "improvement_threshold": 0.01,   # accuracy delta to count as improvement
        "seed": 42,                      # subsampling seed (reproducibility)
        "max_failures": 15,              # failure examples fed to LLM candidate generation
        # --- Escalation ---
        "degradation_threshold": 0.4,    # fraction of degraded queries to trigger escalation
        "backend_warning_threshold": 2,  # degradation resets before backend advisory
        "enable_l2": True,               # L2 refine_context on escalation
        "enable_l3": True,               # L3 modify_plan on L2 stall
        "l2_patience": 2,               # L2 stalls before L3
        "l3_patience": 1,               # L3 stalls before stop
        "l2_temperature": 0.3,           # LLM temperature for L2 transitions
        "l3_temperature": 0.5,           # LLM temperature for L3 transitions
        # --- Critique ---
        "enable_critique": True,         # critique agent between generate/evaluate
    },
    "eval_llm": {
        # "model":       "moonshotai/kimi-k2-instruct-0905",  # 10x more expensive
        "model":       "openai/gpt-oss-120b",
        "provider":    "groq",
        "temperature": 0.4,
        # --- Anthropic (cost: opus >> sonnet >> haiku) ---
        # "model": "claude-opus-4-6",
        # "model": "claude-sonnet-4-6",
        # "model": "claude-haiku-4-5-20251001",
        "max_tokens": 2000,
    },
    "pipeline_params": None        # set by configure_pipeline()
}

# --- Pipeline snapshot & params ---
pipeline_config_full = await show_pipeline_snapshot(svc)
pipeline_params = configure_pipeline(svc, campaign_config)

In [2]:
#@title Load data & evaluation context
RUN_BASELINE = False

train_data, session_terms = prepare_datasets(
    svc["store"], svc["backend_id"],
    excel_path=r"C:\Users\dsacc\Desktop\project-TermNorm\OneDrive_2025-07-02\Austausch Beispiele\Prozessnamen\BOM-example.xlsx",
)
svc["session_terms"] = session_terms

baseline_ps, eval_data, campaign_rounds, baseline_results = await prepare_eval_context(
    svc, train_data, campaign_config, run_baseline=RUN_BASELINE,
)


  Train              : 984 queries
  Test (processes)   : 82 queries
  Test (material)    : 165 queries
  ------------------------------------------------
  Combined queries   : 820 (deduplicated)
  Session identifiers: 94 unique targets

Evaluation data: 984 queries


In [3]:
#@title Experiment dashboard
EXPERIMENT_ID = None  # Set to hex ID to resume (e.g. '68e2c5')

pipeline_params = show_experiment_dashboard(
    svc=svc, experiment_id=EXPERIMENT_ID,
    campaign_config=campaign_config, eval_data=eval_data,
    baseline_prompt_fields=campaign_rounds[0]["prompt_fields"].model_dump() if campaign_rounds else None,
)


  EXPERIMENT DASHBOARD (termnorm-local)
  Dataset runs: 111 total (2 77e7e77777e7, 1 be8c4d8229f6, 1 dbb05e948b53, 107 other)
  Best result: 66.7% (scan)

  Campaigns (most recent first):
    cycle_be8c4d8229f6  interrupted  0 rounds  best=33.3%  base=33.3%
      patience=2  rounds=None  sample=15  model=openai/gpt-oss-120b

    cycle_dbb05e948b53  interrupted  0 rounds  best=33.3%  base=33.3%
      patience=2  rounds=None  sample=15  model=moonshotai/kimi-k2-instruct-09

  Set experiment_id="<short_id>" to see full config and diff
  Active: cycle_13e908d28487



## 3. Explore

Exploration via **Smart Search** (scan advisor + sensitivity scan).

In [4]:
#@title Task context + scan advisor
ADVISOR_MODEL = "openai/gpt-oss-120b"  # model for scan advisor LLM call

task_context = await decompose_task_context(TASK_DESCRIPTION, campaign_config, svc)

# preview_advisor_prompt(campaign_config, svc, task_description=task_context, raw=True)
advisory, scan_variants, schema_labels = await run_scan_advisor(
    campaign_config, svc,
    task_description=task_context,
    model=ADVISOR_MODEL,
)

TASK CONTEXT DECOMPOSITION
--------------------------------------------------
  domain: Life Cycle Assessment (LCA) terminology normalization
  pipeline_purpose: Normalize user‑provided material, process, and geographic descriptors to exact entries in LCA databases (e.g., ecoinvent, GaBi) and flag inputs with no viable match.
  data_characteristics: Free‑form textual inputs containing industrial codes, standards, brand/trade names, chemical shorthand, agricultural qualifiers, geographic tags; mixed German/French/English; includes units and quantities; high variability and low volume per query.
  optimization_goals: Improve profile schema quality and web‑search relevance; increase matching accuracy (precision/recall) across synonyms, standards, and geographic variants; reduce false positives; correctly identify no‑match cases.
  key_challenges: Structural mismatch between unstructured shorthand and rigid database naming; need for domain knowledge to decode brands, standards, and alloy c

2026-03-26 10:05:36 WARNING  [api.services.search.scan_advisor] Scan advisor response truncated (finish_reason=length, max_tokens=2000). Increase max_tokens in eval_llm config.


PRIORITY AXES (ranked by importance)
----------------------------------------
  1. [HIGH] fuzzy_threshold (pipeline_param) -- step: fuzzy_matching
     Controls match strictness for shorthand terms
     Values: ['0.7', '0.85']
  2. [MEDIUM] fuzzy_scorer (pipeline_param) -- step: fuzzy_matching
     Different algorithms affect synonym detection
     Values: ['ratio', 'partial_ratio']
  3. [HIGH] query_prefix (pipeline_param) -- step: web_search
     Guides search toward LCA‑relevant sources
     Values: ['LCA material', 'ecoinvent term']
  4. [MEDIUM] query_suffix (pipeline_param) -- step: web_search
     Adds domain context to results
     Values: ['ecoinvent database', 'life cycle assessment']
  5. [MEDIUM] content_char_limit (pipeline_param) -- step: web_search
     More characters capture richer specifications
     Values: ['1200', '2000']
  6. [MEDIUM] max_sites (pipeline_param) -- step: web_search
     Broader site pool improves coverage
     Values: ['10', '15']
  7. [HIGH] profi

In [5]:
#@title Scan variant config (edit suggested values or add your own)
# Schema axes: mutation tuples ("-", path), ("+", path, type, req, desc),
# ("~", old, new, type, req, desc). Non-schema axes: plain value lists.

scan_sample_size = 3  # queries per scan variant (0 = use all)

scan_variants = {
    # ── Token matching ───────────────────────────────────────────────────
    'max_token_candidates': [10, 30, 50],
    # ── Web search: query framing ────────────────────────────────────────
    'query_prefix': [
        # --- Database-oriented ---
        'ecoinvent', 'GaBi', 'ecoinvent LCA database material name',
        'material composition LCA', # 'ecoinvent equivalent', 'ecoinvent match for', 'identify LCA material for',
        'what is',# 'define', 'identify', 'describe material',
        # 'chemical composition of', 'CAS number', 'IUPAC name for',
        'technical data sheet', 'product specification', # 'manufacturer datasheet', 'material safety data sheet',
        'manufactured from', 'production process for', # 'raw material for',
        # --- Synonym / translation ---
        # 'also known as', # 'synonym for', 'equivalent material', 'alternative name for',
        # 'wikipedia', # 'material properties of',
    ],
    # ── Web search: volume knobs ─────────────────────────────────────────
    'max_sites': [3, 7, 12],
    'num_results': [5, 20, 40],
    'content_char_limit': [400, 800, 1500],
    # ── Fuzzy matching ───────────────────────────────────────────────────
    # 'fuzzy_threshold': [50, 70, 90],
    # 'fuzzy_scorer': ['ratio', 'WRatio', 'token_set_ratio'],
    # ── Entity profiling: LLM tuning ─────────────────────────────────────
    'profiling_temperature': [0.0, 0.3, 0.7],
    'raw_content_limit': [1000, 2500, 8000],
    # ── Prompt fields ────────────────────────────────────────────────────
    'thinking_style': [
          'Think step-by-step: isolate distinguishing features → compare each candidate → assign scores.',
          'Consider the most likely interpretation first, then check alternatives.',       
          'Reason by elimination: discard obviously wrong candidates, then rank the rest.',
      ],
    # ── Entity profiling: schema mutations ───────────────────────────────
    'profiling_schema': [
        # --- LCA-database-specific (original) ---
        [['+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA'],
         ['+', 'database_format_hint', 'string', False, "Best-guess ecoinvent-style name fragment for this entity, e.g. 'market for polyethylene, high density'"]],
        [['+', 'lca_database_names', 'array', True, "Likely ecoinvent or GaBi database entry names that would match this entity, using standard LCA database naming conventions like 'market for X | X | cut-off, U'"]],
        [['-', 'manufacturing_processes'], ['-', 'applications'],
         ['+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names for this entity using standard LCA naming conventions']],
        # [['~', 'notes', 'material_category', 'string', True, "The broad LCA material category this entity belongs to, e.g. 'polyethylene', 'brass', 'steel'"]],
        [['-', 'applications'], ['+', 'ecoinvent_candidate_names', 'array', True, 'Exact ecoinvent activity names this entity most likely maps to']],
        [['-', 'manufacturing_processes'], ['+', 'database_search_tokens', 'array', True, 'Optimized search tokens for LCA database lookup including spelling variants']],
        # [['~', 'classification_aliases', 'classification_aliases', 'array', True, 'All valid ecoinvent/GaBi naming variants including geography codes and system model suffixes']],
        # --- Minimalist: strip to core matching signals ---
        [['-', 'applications'], ['-', 'manufacturing_processes'], ['-', 'notes'], ['-', 'technical_specifications']],
        # --- Chemical identity ---
        [['+', 'cas_number', 'string', False, 'CAS registry number if identifiable from context'],
         ['+', 'chemical_formula', 'string', False, 'Chemical formula or molecular structure notation']],
        # --- Trade name decoding ---
        [['~', 'key_properties', 'trade_names', 'array', True, 'Known commercial/trade names and brand names for this material, e.g. Makrolon=polycarbonate, Delrin=POM']],
        # --- Material hierarchy (specific→generic) ---
        [['+', 'material_hierarchy', 'array', False, 'Classification chain from specific to generic, e.g. [Makrolon 2805, polycarbonate, thermoplastic, polymer]']],
        # --- Process-centric (flip perspective from material to process) ---
        [['~', 'applications', 'production_route', 'string', False, 'Primary production/manufacturing route e.g. injection molding, extrusion, casting'],
         ['~', 'notes', 'form_factor', 'string', False, 'Physical form: granulate, sheet, rod, wire, powder, liquid, film']],
        # --- Standards-focused ---
        [['+', 'applicable_standards', 'array', False, 'DIN/ISO/EN/ASTM standards that reference or define this material'],
         ['-', 'applications']],
        # # --- Geography-aware ---
        # [['+', 'supply_chain_geography', 'string', False, 'Most likely geographic origin or market region for this material']],
        # # --- Confidence / ambiguity signal ---
        # [['+', 'confidence_level', 'string', False, 'How confident the model is in the identification: high/medium/low/ambiguous'],
        #  ['+', 'ambiguity_notes', 'string', False, 'What makes this input hard to identify — abbreviation, trade name, multi-material, etc.']],
        # # --- Spelling / language variant boost ---
        # [['+', 'spelling_variants', 'array', False, 'All known spelling variants across EN/DE/FR, e.g. aluminium/aluminum, polyamid/polyamide'],
        #  ['-', 'notes']],
        # --- Werkstoff / alloy code decoding ---
        [['+', 'material_code_decoded', 'string', False, 'Decoded meaning of any material code, Werkstoff number, or alloy designation present in the input'],
         ['+', 'base_material', 'string', False, 'The fundamental base material, e.g. brass, steel, polycarbonate']],
        # --- Functional equivalence ---
        [['~', 'applications', 'functional_unit', 'string', False, 'The functional unit this material serves, e.g. structural plastic, electrical insulation, food-grade packaging'],
         ['+', 'substitutes', 'array', False, 'Materials that could serve the same functional role']],
    ],
}
scan_variants, schema_labels = resolve_scan_variants(scan_variants, svc=svc)

  max_token_candidates: [10, 30, 50]
  query_prefix: ['ecoinvent', 'GaBi', 'ecoinvent LCA database material name', 'material composition LCA', 'what is', 'technical data sheet', 'product specification', 'manufactured from', 'production process for']
  max_sites: [3, 7, 12]
  num_results: [5, 20, 40]
  content_char_limit: [400, 800, 1500]
  profiling_temperature: [0.0, 0.3, 0.7]
  raw_content_limit: [1000, 2500, 8000]
  thinking_style: ['Think step-by-step: isolate distinguishing features → compare each candidate → assign scores.', 'Consider the most likely interpretation first, then check alternatives.', 'Reason by elimination: discard obviously wrong candidates, then rank the rest.']
  profiling_schema: (baseline + 13 mutations)
    [0] (baseline)
    [1] ('+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA'), ('+', 'database_format_hint', 'string', False, 'Best-guess ecoinvent-style name fragment for this e

In [6]:
#@title Run sensitivity scan
scan_baseline_sp, scan_df, axis_profiles = await run_sensitivity_scan(
    baseline_ps, campaign_config, scan_variants, eval_data,
    scan_sample_size=scan_sample_size,
    svc=svc, experiment_id=EXPERIMENT_ID or "",
)

Active steps: cache_lookup, fuzzy_matching, web_search, entity_profiling, token_matching  Excluded: llm_ranking
  Restructured baseline fields (cached):
    persona: You are a candidate evaluation expert.
    task_intent: Rank 20 entity-name candidates by how well they match a target profile and its c...
    problem_description: Given a JSON entity profile and a core concept, score and rank candidate names o...
    instruction: 1) Extract the entity_category and key distinguishing features from the profile....
    thinking_style: Think step-by-step: isolate distinguishing features → compare each candidate → a...
    answer_format: Valid JSON only: { "reasoning": "...", "ranked_candidates": [ { "rank": 1, "cand...
  Search baseline: b730ac2e9e41 (render: 868 chars)
  Historical data: 49 results across 8 unique prompts
  Matching runs (sp_hash): 31, 3 cached results
  Scan variant coverage (31 matching runs):
    max_token_candidates     10→3 ✓  30→3 ✓  50→3 ✓
    query_prefix           

In [7]:
#@title Scan analytics
difficulty_df = show_scan_analytics(scan_df, axis_profiles, svc)

VARIANT LEADERBOARD (all scan combos)


,rank,axis,variant,accuracy,delta,hits/total,errors
0,1,profiling_schema,schema(10 fields),66.7%,+30.0%,2/3,0
1,2,profiling_temperature,0.3,66.7%,+30.0%,2/3,0
2,3,content_char_limit,400,66.7%,+30.0%,2/3,0
3,4,max_sites,12,66.7%,+30.0%,2/3,0
4,5,profiling_schema,schema(11 fields),66.7%,+30.0%,2/3,0
5,6,profiling_schema,schema(11 fields),66.7%,+28.3%,2/3,0
6,7,num_results,20 (baseline),33.3%,-,1/3,0
7,8,num_results,40 (baseline),33.3%,-,1/3,0
8,9,max_token_candidates,10 (baseline),33.3%,-,1/3,0
9,10,max_token_candidates,50 (baseline),33.3%,-,1/3,0



PER-AXIS STATISTICS


,axis,type,variants,mean_acc,std_acc,best_acc,worst_acc,sensitivity,budget
0,max_token_candidates,pipeline_param,3,22.2%,19.2%,33.3%,0.0%,0.300,skip
1,query_prefix,pipeline_param,9,14.8%,17.6%,33.3%,0.0%,0.317,skip
2,max_sites,pipeline_param,3,44.4%,19.2%,66.7%,33.3%,0.317,skip
3,num_results,pipeline_param,3,22.2%,19.2%,33.3%,0.0%,0.300,skip
4,content_char_limit,pipeline_param,3,44.4%,19.2%,66.7%,33.3%,0.300,skip
5,profiling_temperature,pipeline_param,3,33.3%,33.3%,66.7%,0.0%,0.617,high
6,raw_content_limit,pipeline_param,3,22.2%,19.2%,33.3%,0.0%,0.283,skip
7,thinking_style,prompt_field,3,0.0%,0.0%,0.0%,0.0%,0.017,skip
8,profiling_schema,pipeline_param,14,23.8%,27.5%,66.7%,0.0%,0.600,high


QUERY DIFFICULTY (10 queries across 109 scan runs)
  easy: 0 (0%) | discriminating: 5 (50%) | hard: 5 (50%) | error: 0 (0%)



,query,ground_truth,hit_rate,hits/evals,error_rate,classification
0,Copper Wire/cold forming,"Metal working, average for copper product manu...",0.000000,0/4,0.0,hard
1,PC GF10 makrolon material/0,Injection moulding {RoW}| injection moulding |...,0.000000,0/4,0.0,hard
2,PA6/66 Ultramid C3U/molding,Injection moulding {RER}| injection moulding |...,0.000000,0/5,0.0,hard
3,PA 66 25% GF V0 RAL 7012/0,Injection moulding {RoW}| injection moulding |...,0.000000,0/5,0.0,hard
4,EN 10270-3-1.4568\nX7CrNi17-7 (DIN17224 4.4568...,"Sheet rolling, chromium steel {RER}| sheet rol...",0.000000,0/4,0.0,hard
5,SJRG0010-ABS/molding,Injection moulding {RER}| injection moulding |...,0.038095,4/105,0.0,discriminating
6,Kingfa NPG25,Glass fibre reinforced plastic | 75% PA66 25% ...,0.200000,1/5,0.0,discriminating
7,SJRG0013-PA/molding,Injection moulding {RER}| injection moulding |...,0.333333,1/3,0.0,discriminating
8,PA66-GF25 ULTRAMID A3UG5 RAL7035 grey,Glass fibre reinforced plastic | 75% PA66 25% ...,0.431193,47/109,0.0,discriminating
9,Stainless steel EN 10270-3/winding,"Wire drawing, steel {RER}| wire drawing, steel...",0.485714,51/105,0.0,discriminating


In [8]:
#@title Select scan winner & seed campaign
best_sp = seed_campaign_from_scan(
    scan_df, axis_profiles, scan_baseline_sp, scan_variants,
    campaign_rounds, campaign_config,
)

Selected best from 2 improving axes:
  profiling_temperature     best_delta=+30.0%  value_idx=1  acc=66.7%
  profiling_schema          best_delta=+30.0%  value_idx=0  acc=66.7%

Composed winner: sp_hash=93ae5e9f7885
Updated pipeline_params: {'steps': ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching'], 'llm_ranking': '<1 fields>', 'profiling_temperature': 0.3, 'profiling_schema': '<11 fields>'}

Round    Accuracy   Rolling Avg    Trend
  search    33.3%        33.3%  -


## 4. Optimize

Two modes: **Semi-automatic** (feedback cycle with patience-based auto-stop) or **Manual** (one round at a time).

In [9]:
#@title Feedback cycle preflight
scan_context = show_feedback_preflight(
    campaign_rounds, eval_data, campaign_config,
    pipeline_params=pipeline_params,
    scan_df=scan_df,
    axis_profiles=axis_profiles,
    scan_variants=scan_variants,
    difficulty_df=locals().get("difficulty_df"),
)


  FEEDBACK CYCLE PRE-FLIGHT
  Baseline accuracy      : 33.3%
  Baseline prompt        : You are a candidate evaluation expert.

Rank 20 entity-name candidates by how we...
  ------------------------------------------------------------------
  Max rounds             : unlimited
  Candidates per round   : 5
  Queries per eval       : 15 of 984
  Improvement threshold  : 1.0%
  Patience (L1)          : 2 rounds
  L2 (refine context)    : enabled, patience=2
  L3 (modify plan)       : enabled, patience=1
  ------------------------------------------------------------------
  Candidate model        : openai/gpt-oss-120b
  Creativity             : 0.7
  Pipeline               : (default pipeline)
    Excluded             : llm_ranking
  Strategy               : SCAN-AWARE

  ROUND PIPELINE (what happens each round)
  ------------------------------------------------------------------
  1. BASELINE INPUT
     Prompt: You are a candidate evaluation expert.

Rank 20 entity-name candidates by how

In [10]:
#@title Run optimization (feedback cycle)
dev_reload()

campaign_rounds = await run_feedback_cycle_notebook(
    campaign_rounds, eval_data, campaign_config,
    svc=svc, pipeline_params=pipeline_params,
    scan_context=locals().get("scan_context"),
    experiment_id=EXPERIMENT_ID,
    task_context=task_context,
)

  Interrupt of cells can take up to 60 seconds!
  If a dialog pops up, click 'Cancel' and wait 20 seconds.

╔════════════════════════════════════════════════════════════════════╗
║  FEEDBACK CYCLE STARTING                                           ║
╠════════════════════════════════════════════════════════════════════╣
║  Baseline       33.3%                                              ║
║  Max rounds     999            Patience    2                       ║
║  Candidates     5                                                  ║


2026-03-26 10:05:39 INFO     [api.services.campaign.feedback_cycle] Using provided baseline (acc=0.333)
2026-03-26 10:05:39 INFO     [api.services.campaign.campaign_lifecycle] Cycle identity: cycle_be8c4d8229f6


║  Sample size    15 of 984                                          ║
║  Min detectable ±36.2% (α=0.05, 80% power)                         ║
║  Model          openai/gpt-oss-120b                                ║
║  L2 (refine)    enabled            L3 (plan)   enabled             ║
║  Scan context   YES                                                ║
║  Critique       enabled                                            ║
╚════════════════════════════════════════════════════════════════════╝


2026-03-26 10:05:41 INFO     [api.services.obs.observability_logger] Dataset 'termnorm_ground_truth': 728 items registered, 256 duplicates/empty skipped (from 984 input)
2026-03-26 10:05:41 WARNING  [api.services.obs.observability_logger] Skipping Langfuse cloud dataset registration for 984 items (rate-limit risk). Use the dedicated Langfuse sync cell instead.
2026-03-26 10:05:41 INFO     [api.services.campaign.campaign_lifecycle] Registered 728 dataset items for 'termnorm_ground_truth'
2026-03-26 10:05:41 INFO     [api.services.campaign.feedback_cycle] Feedback cycle round 0 (clean=0/999, acc=0.333, stall=0/2)
2026-03-26 10:05:41 INFO     [api.services.campaign.round_execution] Loaded 5 persisted candidates for round 0
2026-03-26 10:05:41 INFO     [api.services.prompt_eval] SP cache: 4 cached queries for 15 eval queries


  ✓ Initialized  cycle=cycle_be8c4d  samples=15  obs=ON
    Starting fresh (no prior rounds for this cycle)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ROUND 1/999                                               patience 0/2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

├─ GENERATE ─────────────────────────────────────────────────────────────┤
│  Current best    33.3%
│  Prompt          You are a candidate evaluation expert.  Rank 20...
│  Candidates      5   Creativity: 0.7   Scan: YES   Critique: NO
│  Model           openai/gpt-oss-120b
│  Scan focus: 2 improving axes [profiling_temperature, profiling_schema]
│  Scan baseline: 33.3%
├────────────────────────────────────────────────────────────────────────┤
  ✓ 5 candidates generated (loaded from disk)
    C1: Lower profiling_temperature slightly below the ... [instruction]
    C2: Switch to a richer schema with 12 fields includ... [instruction]
    C3: Increase max_sites 

...

  [  5]       MISS           PA 66 25% GF V0 RAL 7012/0                     ERR: [SERVER] HTTP 500: Server error '500 Int


2026-03-26 10:05:55 WARNING  [api.services.prompt_eval] backend_reranker_evaluate failed for PA6/66 Ultramid C3U/molding: 


  [  6]       MISS           PA6/66 Ultramid C3U/molding                    -> ERROR
  [  7]  10.4s MISS 11/3    [llm]  PC  GF10  makrolon material/0                  -> Ceramic tile {GLO}| market for cera
                  ⚠ web_search: 5 of 20 fetched URLs returned content (13 filtered: 4×skip_extension, 5×http_403, 3×http_429, 1×too_short; 2 errors)
  [  8]  13.4s MISS 5/5    [llm]  Copper Wire/cold forming                       -> Silver {GLO}| market for silver | C


2026-03-26 10:06:19 INFO     [api.services.prompt_eval] Graceful stop after query 8/15.
2026-03-26 10:06:19 INFO     [api.services.prompt_eval] Per-query reuse: 4/15 queries from prior runs
2026-03-26 10:06:19 INFO     [api.services.prompt_eval] Saved partial run (8/15 queries) for SP 33db05a7
2026-03-26 10:06:19 WARNING  [api.services.campaign.feedback_cycle] Feedback cycle interrupted at round 0. Completed rounds are checkpointed.



╔════════════════════════════════════════════════════════════════════╗
║  INTERRUPTED — stopped by user                                     ║
╠════════════════════════════════════════════════════════════════════╣
║  Rounds       0              Best         33.3% (round search)     ║
║  Stop reason  interrupted                                          ║
║  Resume: re-run this cell -- rounds auto-restore                   ║
║  Cycle ID     cycle_be8c4d8229f6                                   ║
╚════════════════════════════════════════════════════════════════════╝


In [ ]:
#@title 5. Results — summary, save, sync
show_campaign_summary(campaign_rounds)
show_flip_tracking(campaign_rounds)
show_lineage_chain(campaign_rounds)

# --- Persist (T2: below the fold) ---
save_campaign_winner(
    campaign_rounds, campaign_config, svc["store"], svc["backend_id"],
    experiment_id=EXPERIMENT_ID,
)
sync_langfuse(
    svc["store"], svc["backend_id"],
    dataset_name="termnorm_ground_truth",
)